In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# SAMPLING OF BANDLIMITED WHITE NOISE
# ============================================================

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#7a3e8e;
    margin-bottom:8px;
">
Sampling of Bandlimited White Noise
</div>

<div style="margin-bottom:4px;">
Bandlimited white noise has autocorrelation
<b>Rₓₓ(τ) = 2WS₀ sinc(2Wτ)</b>,
with zeros at τ = n/(2W), n ≠ 0.
</div>

<div style="margin-bottom:4px;">
If the sampling period is Tₛ = 1/(2W), all nonzero sampling lags coincide with these zeros, so different samples are uncorrelated.
</div>

<div style="margin-bottom:4px;">
The graphs below show <b>normalized</b> correlation quantities:
ρₓₓ(τ) = Rₓₓ(τ)/Rₓₓ(0).
Since Rₓₓ(0) = 2WS₀, the common factor 2WS₀ cancels and therefore
<b>ρₓₓ(τ) = sinc(2Wτ)</b>.
</div>

<div style="margin-bottom:4px;">
For the same reason, S₀ multiplies every element of the covariance matrix by the same factor but disappears after normalization.
Thus changing S₀ changes the <b>absolute noise power</b>, but not the <b>normalized correlation structure</b>.
</div>

<div>
<b>This notebook:</b> shows how the sampling interval controls correlation between samples, while S₀ controls their absolute power.
</div>

</div>
""")

# ============================================================
# CONTROLS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='155px')

W_slider = FloatSlider(min=2.0, max=20.0, step=1.0, value=8.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

S0_slider = FloatSlider(min=0.5, max=2.0, step=0.25, value=1.0, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

ratio_slider = FloatSlider(min=0.50, max=1.50, step=0.05, value=1.00, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

K_slider = IntSlider(min=5, max=15, step=2, value=9, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

# ============================================================
# CURRENT VALUES
# ============================================================

W_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">8</div>')

S0_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>')

ratio_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>')

K_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">9</div>')

# ============================================================
# UPDATE CURRENT VALUES
# ============================================================

def update_W(change):
    W_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{W_slider.value:.0f}</div>'

def update_S0(change):
    S0_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{S0_slider.value:.2f}</div>'

def update_ratio(change):
    ratio_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{ratio_slider.value:.2f}</div>'

def update_K(change):
    K_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{K_slider.value}</div>'

W_slider.observe(update_W, names='value')
S0_slider.observe(update_S0, names='value')
ratio_slider.observe(update_ratio, names='value')
K_slider.observe(update_K, names='value')

# ============================================================
# LABELS
# ============================================================

W_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Bandwidth W:</div>')

S0_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">PSD level S₀:</div>')

ratio_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Tₛ / TNyq:</div>')

K_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Samples:</div>')

# ============================================================
# CONTROLS GRID
#
# Row 1: Bandwidth W | PSD level S0
# Row 2: Ts/TNyq     | Samples
# ============================================================

controls_grid = GridBox(
    children=[
        W_label, W_slider, W_value,
        S0_label, S0_slider, S0_value,
        ratio_label, ratio_slider, ratio_value,
        K_label, K_slider, K_value
    ],
    layout=Layout(
        width='800px',
        grid_template_columns='115px 155px 55px 115px 155px 55px',
        grid_template_rows='34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#7a3e8e;
            margin-bottom:5px;
        ">
        Sampling Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='825px',
        padding='10px 14px',
        border='1px solid #d7c6df',
        margin='10px 0px 8px 0px',
        overflow='hidden'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN FUNCTION
# ============================================================

def plot_sampling_white_noise(W=8.0, S0=1.0, ratio=1.0, K=9):

    # --------------------------------------------------------
    # CONTINUOUS AUTOCORRELATION
    #
    # Rxx(tau) = 2 W S0 sinc(2 W tau)
    #
    # numpy sinc(x) = sin(pi*x)/(pi*x)
    # --------------------------------------------------------

    tau = np.linspace(-1.0, 1.0, 2500)

    R_absolute = 2.0 * W * S0 * np.sinc(2.0 * W * tau)

    R0 = 2.0 * W * S0

    R_normalized = R_absolute / R0

    # --------------------------------------------------------
    # NYQUIST SAMPLING INTERVAL
    # --------------------------------------------------------

    T_nyq = 1.0 / (2.0 * W)

    Ts = ratio * T_nyq

    # --------------------------------------------------------
    # SAMPLING POSITIONS
    # --------------------------------------------------------

    half = K // 2

    sample_indices = np.arange(-half, half + 1)

    sample_times = sample_indices * Ts

    sample_absolute = 2.0 * W * S0 * np.sinc(2.0 * W * sample_times)

    sample_normalized = sample_absolute / R0

    # ========================================================
    # ABSOLUTE AND NORMALIZED COVARIANCE MATRICES
    # ========================================================

    covariance_absolute = np.zeros((K, K))

    covariance_normalized = np.zeros((K, K))

    for i in range(K):

        for j in range(K):

            lag = (sample_indices[i] - sample_indices[j]) * Ts

            covariance_absolute[i, j] = 2.0 * W * S0 * np.sinc(2.0 * W * lag)

            covariance_normalized[i, j] = covariance_absolute[i, j] / R0

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(figsize=(10.4, 6.4))

    gs = fig.add_gridspec(1, 2, wspace=0.30)

    ax1 = fig.add_subplot(gs[0, 0])

    ax2 = fig.add_subplot(gs[0, 1])

    # ========================================================
    # GRAPH 1:
    # NORMALIZED AUTOCORRELATION + SAMPLING POINTS
    # ========================================================

    ax1.plot(tau, R_normalized, linewidth=2.0, label='Normalized autocorrelation')

    ax1.stem(sample_times, sample_normalized, basefmt=' ')

    ax1.axhline(0, linewidth=0.8)

    ax1.axvline(0, linewidth=0.8, linestyle=':')

    ax1.set_xlim(-1.0, 1.0)

    ax1.set_ylim(-0.35, 1.15)

    ax1.set_xlabel('Lag τ', fontsize=11)

    ax1.set_ylabel('Normalized Rₓₓ(τ)', fontsize=11)

    ax1.set_title('Sampling the Autocorrelation Function', fontsize=13, pad=9)

    ax1.tick_params(axis='both', labelsize=9)

    ax1.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 2:
    # NORMALIZED COVARIANCE MATRIX
    # ========================================================

    image = ax2.imshow(covariance_normalized, vmin=-1.0, vmax=1.0, origin='upper', aspect='equal')

    ax2.set_xlabel('Sample index j', fontsize=11)

    ax2.set_ylabel('Sample index i', fontsize=11)

    ax2.set_title('Normalized Covariance Matrix', fontsize=13, pad=9)

    ax2.tick_params(axis='both', labelsize=9)

    fig.colorbar(image, ax=ax2, fraction=0.046, pad=0.04)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(left=0.08, right=0.95, top=0.90, bottom=0.10)

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL RESULTS
    # ========================================================

    normalized_off_diagonal = covariance_normalized - np.diag(np.diag(covariance_normalized))

    absolute_off_diagonal = covariance_absolute - np.diag(np.diag(covariance_absolute))

    max_normalized_off_diagonal = np.max(np.abs(normalized_off_diagonal))

    max_absolute_off_diagonal = np.max(np.abs(absolute_off_diagonal))

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial;
        font-size:15px;
        line-height:1.45;
        width:930px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Nyquist interval:</b>
    TNyq = 1/(2W) = {T_nyq:.5f}

    &nbsp;&nbsp;&nbsp;

    <b>Current sampling interval:</b>
    Tₛ = {Ts:.5f}

    <br>

    <b>Absolute variance / noise power:</b>
    Rₓₓ(0) = 2WS₀ = {R0:.5f}

    <br>

    <b>Maximum normalized off-diagonal covariance:</b>
    {max_normalized_off_diagonal:.5f}

    &nbsp;&nbsp;&nbsp;

    <b>Maximum absolute off-diagonal covariance:</b>
    {max_absolute_off_diagonal:.5f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_sampling_white_noise,
    {
        'W': W_slider,
        'S0': S0_slider,
        'ratio': ratio_slider,
        'K': K_slider
    }
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial;
    font-size:15px;
    line-height:1.45;
    width:1050px;
    padding:11px 15px;
    border:1px solid #d8cbe3;
    background:#fcf9ff;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#7a3e8e;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
For Tₛ = 1/(2W), every nonzero sampling lag coincides with a zero of the sinc autocorrelation, so the covariance matrix becomes diagonal.
</div>

<div style="margin-bottom:4px;">
Changing Tₛ away from this interval moves the sample positions away from the zeros and therefore introduces nonzero correlation between different samples.
</div>

<div style="margin-bottom:4px;">
Changing S₀ multiplies Rₓₓ(τ) and every covariance-matrix element by the same factor 2WS₀.
After normalization, this common factor appears in both numerator and denominator and cancels.
</div>

<div>
Therefore S₀ changes the <b>absolute variance and covariance</b>, but does not change the <b>normalized autocorrelation curve or the normalized covariance pattern</b> shown in the figures.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        controls_card,
        output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)